In [1]:
# import dependencies
import pandas as pd
import numpy as np
import re
from rapidfuzz import process

In [2]:
# load acs data (for merging town names, GEOID, population and housing unit data)
acs_df = pd.read_csv("../data/cleaned/acs_summary.csv")

In [3]:
# load policies data (clean and merge after claims data)
policies = pd.read_csv("../data/raw/NFIP/vt_nfip_policies.csv", low_memory=False)

In [4]:
# open vt nfip claims data
claims = pd.read_csv("../data/raw/NFIP/vt_nfip_claims.csv")
claims.head()

,agricultureStructureIndicator,asOfDate,basementEnclosureCrawlspaceType,policyCount,crsClassificationCode,dateOfLoss,elevatedBuildingIndicator,elevationCertificateIndicator,elevationDifference,baseFloodElevation,...,rentalPropertyIndicator,state,reportedCity,reportedZipCode,countyCode,censusTract,censusBlockGroupFips,latitude,longitude,id
0,0,2026-02-11T00:00:00.000Z,2.0,1,NaN,2000-12-17T00:00:00.000Z,0,1,NaN,NaN,...,0,VT,Currently Unavailable,5602.0,50023.0,5.002395e+10,5.002395e+11,44.3,-72.6,c97f77cf-d367-47bf-be38-1cee821b5629
1,0,2026-02-11T00:00:00.000Z,2.0,1,NaN,2011-05-26T00:00:00.000Z,0,NaN,NaN,NaN,...,0,VT,Currently Unavailable,5602.0,50023.0,5.002395e+10,5.002395e+11,44.3,-72.6,085f146c-74c3-47e4-8b4c-60838f1faf0f
2,0,2026-02-11T00:00:00.000Z,1.0,1,NaN,1992-03-11T00:00:00.000Z,0,NaN,NaN,NaN,...,0,VT,Currently Unavailable,5602.0,50023.0,5.002395e+10,5.002395e+11,44.3,-72.6,a48611f1-1e92-4e38-972c-ec42a556a8e1
3,0,2026-02-11T00:00:00.000Z,NaN,1,NaN,1981-02-19T00:00:00.000Z,0,NaN,NaN,NaN,...,0,VT,Currently Unavailable,5476.0,50011.0,5.001101e+10,5.001101e+11,45.0,-72.7,65627989-987c-4916-bf1d-acc450ddd8e6
4,0,2026-02-11T00:00:00.000Z,NaN,1,NaN,2011-08-28T00:00:00.000Z,1,NaN,NaN,NaN,...,0,VT,Currently Unavailable,5748.0,50001.0,5.000196e+10,5.000196e+11,43.9,-72.9,d3eb3b79-d6de-473c-9939-535228b538d1


In [5]:
# manual dict and regex function to clean town names in claims and policies data for merging with acs_df

# manual corrections for town names in claims and policies data
manual_town_dict = {
    # spelling and punctuation errors
    "ENOSBURG TOWN": "ENOSBURGH TOWN",
    "FERRISBURG TOWN": "FERRISBURGH TOWN",
    "MORRISTOWNTOWN": "MORRISTOWN TOWN",
    "MT. HOLLY TOWN": "MOUNT HOLLY TOWN",
    "ST ALBANS TOWN": "ST. ALBANS TOWN",
    "HARDWICK TOWN AND TOWN": "HARDWICK TOWN",
    "NEWFANE TOWN AND TOWN": "NEWFANE TOWN",
    # villages inside of towns
    "MORRISVILLE TOWN": "MORRISTOWN TOWN",
    "JEFFERSONVILLE TOWN": "CAMBRIDGE TOWN",
    "ORLEANS TOWN": "BARTON TOWN",
    # additional manual corrections, for towns in policies data
    "ENOSBURG FALLS TOWN": "ENOSBURGH TOWN",
    "ESSEX JCT. TOWN": "ESSEX JUNCTION CITY",
    "BELLOWS FALLS TOWN": "ROCKINGHAM TOWN",
    "NORTH BENNINGTON TOWN": "BENNINGTON TOWN",
    "OLD BENNINGTON TOWN": "BENNINGTON TOWN",
}


# function to regex names
def clean_community(name):
    if pd.isna(name):
        return name
    name = name.upper()
    name = re.sub(r",", "", name)
    name = re.sub(r" OF", "", name)
    # remove 'VILLAGE', which conveniently solves some of the mismatches
    # want to keep 'TOWN' and 'CITY', sorts the Rutland-Barre-Newport-St Albans city/town mess
    name = re.sub(r"VILLAGE", "TOWN", name)
    return name.strip()

## Clean and add Claims data

In [6]:
# clean nfipCommunityName to match town names in acs_df for eventual merging

# regex clean the names first, then apply manual corrections
claims["nfipCommunityName_clean"] = claims["nfipCommunityName"].apply(clean_community)
claims["nfipCommunityName_clean"] = claims["nfipCommunityName_clean"].replace(
    manual_town_dict
)
claims["nfipCommunityName_clean"].value_counts()

nfipCommunityName_clean
MONTPELIER CITY    208
BARRE CITY         206
WATERBURY TOWN     132
LUDLOW TOWN         68
RUTLAND CITY        54
                  ... 
ST. ALBANS CITY      1
STAMFORD TOWN        1
WINHALL TOWN         1
BARNARD TOWN         1
HINESBURG TOWN       1
Name: count, Length: 160, dtype: int64

In [7]:
# bin funding periods

# filter to match funding analysis period (FEMA HMA data starts in 1989)
claims = claims[claims["yearOfLoss"] >= 1989]


# bin years to match funding buckets
def claims_period(year):
    if pd.isna(year):
        return "unknown"
    if year < 2011:
        return "pre_2011_nfip_claims"
    elif year < 2023:
        return "2011_2022_nfip_claims"
    else:
        return "2023_plus_nfip_claims"


# apply function to create claims_period column
claims["claims_period"] = claims["yearOfLoss"].apply(claims_period)
claims["claims_period"].value_counts()

claims_period
2011_2022_nfip_claims    1724
2023_plus_nfip_claims     910
pre_2011_nfip_claims      691
Name: count, dtype: int64

In [8]:
# sum amount paid in insurance claims
claims["total_paid"] = (
    claims["amountPaidOnBuildingClaim"].fillna(0)
    + claims["amountPaidOnContentsClaim"].fillna(0)
    + claims["amountPaidOnIncreasedCostOfComplianceClaim"].fillna(0)
)

# aggregate count and dollar amount of claims by town name
claims_summary = (
    claims.groupby("nfipCommunityName_clean")
    .agg(nfip_claims=("id", "count"), total_nfip_claims_paid=("total_paid", "sum"))
    .reset_index()
)
claims_summary.head()

,nfipCommunityName_clean,nfip_claims,total_nfip_claims_paid
0,ANDOVER TOWN,3,38465.70
1,ARLINGTON TOWN,5,44870.80
2,BARNARD TOWN,1,0.00
3,BARNET TOWN,4,103197.02
4,BARRE CITY,206,9554298.45


In [9]:
# bin amount paid and count of claims by funding period

# pivot to get total claims paid per period as columns
claims_by_period = (
    claims.pivot_table(
        index="nfipCommunityName_clean",
        columns="claims_period",
        values="total_paid",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
    .rename_axis(None, axis=1)
)

# pivot for count of claims per period
num_claims_by_period = (
    claims.pivot_table(
        index="nfipCommunityName_clean",
        columns="claims_period",
        values="id",
        aggfunc="count",
        fill_value=0,
    )
    .reset_index()
    .rename_axis(None, axis=1)
)

# merge these back into claims_summary
claims_summary = claims_summary.merge(
    claims_by_period, on="nfipCommunityName_clean", how="left"
)
claims_summary = claims_summary.merge(
    num_claims_by_period,
    on="nfipCommunityName_clean",
    how="left",
    suffixes=("", "_count"),
)
claims_summary.shape

(160, 9)

In [10]:
# merge aggregated funding data back to town-level acs data on population and GEOID, to get a complete town-level dataset for analysis

# create town_name_upper in acs_df for merging
acs_df["town_name_upper"] = acs_df["town_name"].str.upper().str.strip()

# check for unmatched town names in claims_summary that are not in acs_df
unmatched = claims_summary[
    ~claims_summary["nfipCommunityName_clean"].isin(acs_df["town_name_upper"])
]
print(
    f'Unmatched towns (should be empty if successful):\n{unmatched["nfipCommunityName_clean"]}'
)

# left join, to exclude any towns in claims_summary that don't match to acs_df
# redundant safety check after unmatched (immediately above)
nfip_df = pd.merge(
    acs_df[
        [
            "GEOID",
            "town_name",
            "town_name_upper",
            "total_population",
            "occupied_housing_units",
        ]
    ],
    claims_summary,
    left_on="town_name_upper",
    right_on="nfipCommunityName_clean",
    how="left",
)

# fill remaining NaN values with 0
# numeric_cols = nfip_df.select_dtypes(include=[np.number]).columns
# nfip_df[numeric_cols] = nfip_df[numeric_cols].fillna(0)

# drop helper columns
nfip_df = nfip_df.drop(columns=["town_name_upper", "nfipCommunityName_clean"])

# checking for 256 towns
nfip_df.shape

Unmatched towns (should be empty if successful):
Series([], Name: nfipCommunityName_clean, dtype: object)


(256, 12)

In [11]:
# calculate claims per capita and claims per housing unit
nfip_df["claims_paid_per_capita"] = np.where(
    nfip_df["total_population"] > 0,
    nfip_df["total_nfip_claims_paid"] / nfip_df["total_population"],
    np.nan,
)

nfip_df["claims_paid_per_housing_unit"] = np.where(
    nfip_df["occupied_housing_units"] > 0,
    nfip_df["total_nfip_claims_paid"] / nfip_df["occupied_housing_units"],
    np.nan,
)

In [12]:
print(nfip_df.shape)
display(nfip_df.head())

(256, 14)


,GEOID,town_name,total_population,occupied_housing_units,nfip_claims,total_nfip_claims_paid,2011_2022_nfip_claims,2023_plus_nfip_claims,pre_2011_nfip_claims,2011_2022_nfip_claims_count,2023_plus_nfip_claims_count,pre_2011_nfip_claims_count,claims_paid_per_capita,claims_paid_per_housing_unit
0,5000100325,Addison town,1175,490,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5001900475,Albany town,934,394,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5001300860,Alburgh town,1832,786,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5002701300,Andover town,616,238,3.0,38465.7,9285.32,0.0,29180.38,1.0,1.0,1.0,62.444318,161.620588
4,5000301450,Arlington town,2598,864,5.0,44870.8,44870.80,0.0,0.00,5.0,0.0,0.0,17.271286,51.933796


## Clean and add Policies data

In [13]:
# clean nfipCommunityName in policies data to match town names in acs_df for merging
# use the cleaning function and manual dict near the top of the notebook
policies["nfipCommunityName_clean"] = policies["nfipCommunityName"].apply(
    clean_community
)
policies["nfipCommunityName_clean"] = policies["nfipCommunityName_clean"].replace(
    manual_town_dict
)
print("Total policies:", len(policies))
policies["nfipCommunityName_clean"].value_counts(dropna=False)

Total policies: 64798


nfipCommunityName_clean
NaN                16468
MONTPELIER CITY     3175
BENNINGTON TOWN     2184
BARRE CITY          2021
WATERBURY TOWN      1873
                   ...  
WESTFORD TOWN          9
SANDGATE TOWN          5
WOODFORD TOWN          3
TINMOUTH TOWN          2
KIRBY TOWN             1
Name: count, Length: 219, dtype: int64

I'm losing a quarter of policies (NaN nfipCommunityName_clean). Doesn't matter for my 2023 and today time bin, but I lose 2/3 of 2011 Hurricane Irene time bin. Fixing it using zip codes to assing a town name to the NaN's.

In [14]:
# opening a csv of US zip code data, and creating a manual dict and name matching function
# to clean NaN town names in policies data for merging with acs_df

# open zip code data for merging with policies data
zip_codes = pd.read_csv("../data/resources/zip_code_database.csv")

# filter zip code data to Vermont only
vt_zips = zip_codes[zip_codes["state"] == "VT"].copy()

# manual mapping dict for the largest unmatched-by-zip-code places in policies data
# this gets me down to 200, out of 3641, so I'm calling it
village_to_town = {
    "SAINT JOHNSBURY TOWN": "ST. JOHNSBURY TOWN",
    "SAINT ALBANS TOWN": "ST. ALBANS TOWN",
    "LYNDONVILLE TOWN": "LYNDON TOWN",
    "MANCHESTER CENTER TOWN": "MANCHESTER TOWN",
    "WEST DOVER TOWN": "DOVER TOWN",
    "WHITE RIVER JUNCTION TOWN": "HARTFORD TOWN",
    "CUTTINGSVILLE TOWN": "SHREWSBURY TOWN",
    "MONTGOMERY CENTER TOWN": "MONTGOMERY TOWN",
    "PROCTORSVILLE TOWN": "PROCTOR TOWN",
    "SAINT ALBANS BAY TOWN": "ST. ALBANS TOWN",
}


# clean town names to match nfipCommunityName_clean convention (uppercase, add TOWN or CITY)
def assign_town_city(row):
    # standardize town name to uppercase and strip whitespace
    town = row["primary_city"].upper().strip()
    # there are ten cities in Vermont
    # these four are city/town pairs that bedevil me
    if town in ["NEWPORT", "ST. ALBANS", "BARRE", "RUTLAND"]:
        # these are the city names based on zip code data
        city_zip_codes = {
            "NEWPORT": ["05855"],
            "ST. ALBANS": ["05478"],
            "BARRE": ["05641"],
            "RUTLAND": ["05701"],
        }
        if str(row["zip"]) in city_zip_codes[town]:
            return f"{town} CITY"
        else:
            return f"{town} TOWN"
    # handle the remaining six cities
    elif town in [
        "VERGENNES",
        "BURLINGTON",
        "SOUTH BURLINGTON",
        "ESSEX JUNCTION",
        "WINOOSKI",
        "MONTPELIER",
    ]:
        return f"{town} CITY"
    # towns
    else:
        # check if the town is in the village_to_town mapping
        key = f"{town} TOWN"
        if key in village_to_town:
            return village_to_town[key]
        # return the town name with TOWN suffix if no other conditions are met
        return f"{town} TOWN"

In [15]:
# clean city names in ZIP code data, find NaN nfipCommunityName_clean values in policies data,
# and merge on ZIP code to fill missing town names in policies data

# apply function to create nfipCommunityName_clean_zip column in vt_zips
vt_zips["nfipCommunityName_clean_zip"] = vt_zips.apply(assign_town_city, axis=1)

# merge policies with missing town on ZIP code
# only update where nfipCommunityName_clean is null/NaN and reportedZipCode is present
mask = policies["nfipCommunityName_clean"].isna() & policies["reportedZipCode"].notna()
policies_missing = policies.loc[mask].copy()

# ensure zip code is string and zero-padded to 5 digits
vt_zips["zip"] = vt_zips["zip"].astype(str).str.zfill(5)
# convert reportedZipCode to integer (drop .0), then zero-pad to 5 digits
policies_missing["reportedZipCode"] = (
    policies_missing["reportedZipCode"]
    .astype(int)  # drop .0, before converting to string
    .astype(str)
    .str.zfill(5)
)

# merge on ZIP code
policies_missing = policies_missing.merge(
    vt_zips[["zip", "nfipCommunityName_clean_zip"]],
    left_on="reportedZipCode",
    right_on="zip",
    how="left",
)


# fill missing nfipCommunityName_clean with the cleaned value from ZIP
policies.loc[mask, "nfipCommunityName_clean"] = policies_missing[
    "nfipCommunityName_clean_zip"
].values

# print diagnostics
print(
    "nfipCommunityName_clean now filled, after using ZIP code:",
    policies["nfipCommunityName_clean"].notna().sum(),
)
print(
    "Still missing after ZIP merge:", policies["nfipCommunityName_clean"].isna().sum()
)

nfipCommunityName_clean now filled, after using ZIP code: 64755
Still missing after ZIP merge: 43


In [16]:
# aggregate policy counts by major flooding events and today

# reference dates
irene_date = pd.Timestamp("2011-08-28").tz_localize("UTC")
flood2023_date = pd.Timestamp("2023-07-10").tz_localize(
    "UTC"
)  # actual flood dates were July 9-12, esp. 10-11
today = pd.Timestamp.today().normalize().tz_localize("UTC")

# convert date columns are datetime
policies["policyEffectiveDate"] = pd.to_datetime(policies["policyEffectiveDate"])
policies["policyTerminationDate"] = pd.to_datetime(policies["policyTerminationDate"])
policies["cancellationDateOfFloodPolicy"] = pd.to_datetime(
    policies["cancellationDateOfFloodPolicy"]
)

# compute true end date of policy
policies["policyEndDate"] = policies[
    ["policyTerminationDate", "cancellationDateOfFloodPolicy"]
].min(axis=1)


# function to get active policies at a given date
def get_active_policies_at(date):
    return policies[
        (policies["policyEffectiveDate"] <= date)
        & (policies["policyEndDate"] >= date)
        & (policies["policyCount"] > 0)
    ]


# aggregate for each date
active_irene = (
    get_active_policies_at(irene_date)
    .groupby("nfipCommunityName_clean")
    .agg(irene_policies=("policyCount", "sum"))
    .reset_index()
)
active_2023 = (
    get_active_policies_at(flood2023_date)
    .groupby("nfipCommunityName_clean")
    .agg(flood2023_policies=("policyCount", "sum"))
    .reset_index()
)
active_today = (
    get_active_policies_at(today)
    .groupby("nfipCommunityName_clean")
    .agg(today_policies=("policyCount", "sum"))
    .reset_index()
)

# print totals and non-null town counts for each date
print("Total active policies at Irene date:", len(get_active_policies_at(irene_date)))
print(
    "Total active policies at 2023 flood date:",
    len(get_active_policies_at(flood2023_date)),
)
print("Total active policies today:", len(get_active_policies_at(today)))
print(
    "Active policies with non-null town at Irene date:",
    get_active_policies_at(irene_date)["nfipCommunityName_clean"].notnull().sum(),
)
print(
    "Active policies with non-null town at 2023 flood date:",
    get_active_policies_at(flood2023_date)["nfipCommunityName_clean"].notnull().sum(),
)
print(
    "Active policies with non-null town today:",
    get_active_policies_at(today)["nfipCommunityName_clean"].notnull().sum(),
)

# merge all three on town
policies_summary = active_irene.merge(
    active_2023, on="nfipCommunityName_clean", how="outer"
).merge(active_today, on="nfipCommunityName_clean", how="outer")
print(policies_summary.shape)
policies_summary.head()

Total active policies at Irene date: 3621
Total active policies at 2023 flood date: 2944
Total active policies today: 3454
Active policies with non-null town at Irene date: 3614
Active policies with non-null town at 2023 flood date: 2944
Active policies with non-null town today: 3454
(275, 4)


,nfipCommunityName_clean,irene_policies,flood2023_policies,today_policies
0,ADDISON TOWN,NaN,NaN,1.0
1,ALBURGH TOWN,15.0,13.0,15.0
2,ANDOVER TOWN,2.0,5.0,6.0
3,ARLINGTON TOWN,20.0,25.0,21.0
4,BAKERSFIELD TOWN,2.0,1.0,1.0


Above, note the count of Irene non-null towns. Was 1391 (out of 3621) before matching zip codes. 2023 and today always completely matched.

In [17]:
# create upper case town name for merging
nfip_df["town_name_upper"] = nfip_df["town_name"].str.upper().str.strip()

# check for unmatched town names in policies_summary that are not in nfip_df
unmatched_policies = policies_summary[
    ~policies_summary["nfipCommunityName_clean"].isin(nfip_df["town_name_upper"])
]
print(f"Unmatched towns: {len(unmatched_policies)}")
print(f'Total unmatched policies: {unmatched_policies["irene_policies"].sum()}')
# print(f"Unmatched towns (should be empty if successful):\n{unmatched_policies}")

# merge with nfip_df
nfip_df = nfip_df.merge(
    policies_summary,
    left_on="town_name_upper",
    right_on="nfipCommunityName_clean",
    how="left",
)


# fill remaining NaN values with 0
# numeric_cols = nfip_df.select_dtypes(include=[np.number]).columns
# nfip_df[numeric_cols] = nfip_df[numeric_cols].fillna(0)

# drop helper columns
nfip_df = nfip_df.drop(columns=["town_name_upper", "nfipCommunityName_clean"])

# checking for 256 towns
nfip_df.shape

Unmatched towns: 60
Total unmatched policies: 200.0


(256, 17)

I'm not bothering with that five percent of Hurricane Irene policy count number. Uncomment the below to see value counts and town names.

In [18]:
# print(unmatched_policies[["nfipCommunityName_clean", "irene_policies"]].sort_values("irene_policies", ascending=False))

In [19]:
# calculate proportion of housing units currently with insurance, or insurance penetration
nfip_df["current_insurance_penetration"] = np.where(
    nfip_df["occupied_housing_units"] > 0,
    nfip_df["today_policies"] / nfip_df["occupied_housing_units"],
    np.nan,
)

In [20]:
# check that insurance penetration looks reasonable
nfip_df.current_insurance_penetration.describe()

count    205.000000
mean       0.018342
std        0.023454
min        0.000731
25%        0.005792
50%        0.011013
75%        0.024306
max        0.239829
Name: current_insurance_penetration, dtype: float64

In [21]:
# print shape, display head, alphabetize, and export to csv
print(nfip_df.shape)
display(nfip_df.head())
nfip_df = nfip_df.sort_values("town_name")
nfip_df.to_csv("../data/cleaned/nfip_summary.csv", index=False)

(256, 18)


,GEOID,town_name,total_population,occupied_housing_units,nfip_claims,total_nfip_claims_paid,2011_2022_nfip_claims,2023_plus_nfip_claims,pre_2011_nfip_claims,2011_2022_nfip_claims_count,2023_plus_nfip_claims_count,pre_2011_nfip_claims_count,claims_paid_per_capita,claims_paid_per_housing_unit,irene_policies,flood2023_policies,today_policies,current_insurance_penetration
0,5000100325,Addison town,1175,490,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.002041
1,5001900475,Albany town,934,394,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5001300860,Alburgh town,1832,786,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,13.0,15.0,0.019084
3,5002701300,Andover town,616,238,3.0,38465.7,9285.32,0.0,29180.38,1.0,1.0,1.0,62.444318,161.620588,2.0,5.0,6.0,0.025210
4,5000301450,Arlington town,2598,864,5.0,44870.8,44870.80,0.0,0.00,5.0,0.0,0.0,17.271286,51.933796,20.0,25.0,21.0,0.024306
